In [0]:
%sql
CREATE OR REPLACE TABLE  workspace.gold_crashes.dim_crashes
USING DELTA
AS
SELECT 
    CRASH_RECORD_ID AS crash_record_id,
    CRASH_DATE AS crash_date,
    WEATHER_CONDITION AS weather_condition,
    LIGHTING_CONDITION AS lighting_condition,
    DATE_POLICE_NOTIFIED AS date_police_notified,
    INJURIES_TOTAL AS total_injuries,
    INJURIES_FATAL AS fatal_injuries,
    INJURIES_INCAPACITATING AS incapacitating_injuries,
    INJURIES_NON_INCAPACITATING AS non_incapacitating_injuries,
    INJURIES_REPORTED_NOT_EVIDENT AS reported_not_evident_injuries,
    INJURIES_NO_INDICATION AS no_indication_injuries,
    INJURIES_UNKNOWN AS unknown_injuries,
    CRASH_HOUR AS crash_hour,
    CRASH_DAY_OF_WEEK AS crash_day_of_week,
    CRASH_MONTH AS crash_month,
    LATITUDE AS latitude,
    LONGITUDE AS longitude,
      CASE 
        WHEN UPPER(DAMAGE) LIKE '%OVER%1%500%' THEN 2000
        WHEN UPPER(DAMAGE) LIKE '%500%OR%LESS' THEN 500
        WHEN UPPER(DAMAGE) LIKE '%501%' THEN 1500
        ELSE NULL
    END AS damage_cost
FROM workspace.silver_crashes.crashes_table;


In [0]:
%sql
SELECT 
sum(INJURIES_TOTAL) as total_injuries, 
sum(INJURIES_FATAL) as total_fatal_injuries,
date_part('year' ,CRASH_DATE) AS year 
FROM workspace.silver_crashes.crashes_table
GROUP BY year
ORDER BY total_injuries DESC


In [0]:
%sql

SELECT
COUNT(*) AS number_reported_crashes,
date_part('year' ,CRASH_DATE) AS year,
date_format(CRASH_DATE, 'MMMM') AS month
FROM workspace.silver_crashes.crashes_table
GROUP BY year, month
HAVING year BETWEEN 2020 AND 2025
ORDER BY number_reported_crashes DESC




In [0]:
%sql

CREATE OR REPLACE TABLE  workspace.gold_crashes.dim_crashes_location
USING DELTA
AS
SELECT
  ROUND(latitude, 2) AS rounded_latitude,
  ROUND(longitude, 2) AS rounded_longitude,
  COUNT(*) AS accident_count
FROM
  workspace.silver_crashes.crashes_table
WHERE
  latitude != -1
  AND longitude != -1
GROUP BY
  ROUND(latitude, 2),
  ROUND(longitude, 2);

In [0]:
%sql
CREATE OR REPLACE TABLE  workspace.gold_crashes.dim_crash_person_data
USING DELTA
AS
SELECT 
row_number() over (order by PERSON_ID) AS unique_id,
p.CRASH_RECORD_ID as crash_record_id,
p.PERSON_ID as person_id,
p.PERSON_TYPE as person_type,
p.CRASH_DATE as crash_date,
p.SEX as sex,
p.VEHICLE_ID as vehicle_id,
p.INJURY_CLASSIFICATION as injury_class,
p.AGE as age,
v.MAKE as car_make,
v.OCCUPANT_CNT as num_occupants
FROM workspace.silver_crashes.person_table as p
LEFT JOIN workspace.silver_crashes.vehicles_table as v
ON p.VEHICLE_ID = v.VEHICLE_ID
WHERE p.PERSON_TYPE = 'DRIVER' AND p.VEHICLE_ID != 0